Step 1:


In [1]:
pip install openai pydub

Note: you may need to restart the kernel to use updated packages.


Step 2:

In [5]:
import os
print("Notebook folder:", os.getcwd())
print()
print("Files here:", os.listdir("."))

Notebook folder: c:\Users\bbars\Desktop\Week 1\Day 5\lab whisper implementation\lab-whisper-stt

Files here: ['.env', '.git', '.gitignore', 'instructions.md', 'labwhisper.ipynb', 'labwhisper.mp3.mp3', 'README.md', 'rubric.md']


In [7]:
  audio_path = "labwhisper.mp3"

In [9]:
  audio_path = "audio/your_file_name.mp3"

In [10]:
print("hello")

hello


In [11]:
import os
print("Notebook folder:", os.getcwd())
print("Files here:", os.listdir("."))

Notebook folder: c:\Users\bbars\Desktop\Week 1\Day 5\lab whisper implementation\lab-whisper-stt
Files here: ['.env', '.git', '.gitignore', 'instructions.md', 'labwhisper.ipynb', 'labwhisper.mp3.mp3', 'README.md', 'rubric.md']


In [12]:
import os
from pydub import AudioSegment

audio_path = "labwhisper.mp3.mp3"

print("File exists:", os.path.exists(audio_path))

File exists: True


In [13]:
audio = AudioSegment.from_file(audio_path)

duration_seconds = len(audio) / 1000
print(f"Duration:    {duration_seconds:.1f} seconds ({duration_seconds / 60:.1f} minutes)")
print(f"Channels:    {audio.channels}")
print(f"Sample rate: {audio.frame_rate} Hz")
print(f"File size:   {os.path.getsize(audio_path) / (1024 * 1024):.2f} MB")

Duration:    133.0 seconds (2.2 minutes)
Channels:    2
Sample rate: 44100 Hz
File size:   2.03 MB


Step 3:

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()          # reads your API key from the .env file
client = OpenAI()      # uses the OPENAI_API_KEY variable
print("Client ready")

Client ready


In [15]:
short_clip = audio[:30 * 1000]      # pydub slices in milliseconds
short_clip.export("short_clip.mp3", format="mp3")
print("Clip length:", len(short_clip) / 1000, "seconds")

Clip length: 30.0 seconds


In [16]:
with open("short_clip.mp3", "rb") as f:
    response = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )

print(response.text)

Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.


In [17]:
print(type(response))
print(response)

<class 'openai.types.audio.transcription.Transcription'>
Transcription(text="Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.", languages=None, logprobs=None, usage=UsageDuration(seconds=30.0, type='duration'))


Step 4:

In [18]:
prompt_text = (
    "An oral history interview with Miss Davis, an elderly woman, "
    "about being relocated by the government. "
    "She speaks in a rural Southern dialect."
)

In [19]:
with open("short_clip.mp3", "rb") as f:
    prompted_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        prompt=prompt_text
    )

prompted_text = prompted_response.text
print(prompted_text)

Now you just pretend you hadn't talked to me any before and you tell me about the government moving you up here.


In [20]:
with open("transcript_prompted.txt", "w", encoding="utf-8") as f:
    f.write(prompted_text)

# Keep the Step 3 (unprompted) text in a variable too
unprompted_text = response.text
print("Saved. Prompted words:", len(prompted_text.split()),
      "| Unprompted words:", len(unprompted_text.split()))

Saved. Prompted words: 22 | Unprompted words: 76


In [21]:
print(prompted_text)


Now you just pretend you hadn't talked to me any before and you tell me about the government moving you up here.


In [22]:
prompt_text = "Miss Davis, government, garden."

with open("short_clip.mp3", "rb") as f:
    prompted_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        prompt=prompt_text
    )

prompted_text = prompted_response.text
print(prompted_text)
print()
print("Prompted words:", len(prompted_text.split()),
      "| Unprompted words:", len(unprompted_text.split()))

Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.

Prompted words: 76 | Unprompted words: 76


1. The first prompt (the long description) produced a result that dropped most of the speech (22 words vs. 76).
2. The shorter prompt gave [your result here].
3. Conclusion: prompts help most with names and technical terms, and long descriptive prompts can hurt.

Step 5: Transcription Without Prompts (Unguided Approach)

In [23]:
with open("short_clip.mp3", "rb") as f:
    unprompted_response = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )

unprompted_text = unprompted_response.text

with open("transcript_unprompted.txt", "w", encoding="utf-8") as f:
    f.write(unprompted_text)

print(unprompted_text)

Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.


In [24]:
print("UNPROMPTED:")
print(unprompted_text)
print()
print("PROMPTED:")
print(prompted_text)
print()
print("Unprompted words:", len(unprompted_text.split()))
print("Prompted words:  ", len(prompted_text.split()))

UNPROMPTED:
Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.

PROMPTED:
Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me.

Unprompted words: 76
Prompted words:   76


In [25]:
import difflib

unprompted_words = unprompted_text.split()
prompted_words = prompted_text.split()

differences = [d for d in difflib.ndiff(unprompted_words, prompted_words)
               if d.startswith("- ") or d.startswith("+ ")]

print("Differences found:", len(differences))
for d in differences:
    print(d)

similarity = difflib.SequenceMatcher(None, unprompted_text, prompted_text).ratio()
print(f"\nSimilarity: {similarity * 100:.1f}%")

Differences found: 0

Similarity: 100.0%


In [26]:
print(unprompted_text == prompted_text)

True


"For this short, clear audio clip, the prompted and unprompted transcripts were identical (76 words each, 100% similar). This suggests prompts add little when the audio is clear and the vocabulary is simple. However, my first prompt, which was long and descriptive, caused the transcript to be cut to 22 words, so a poorly chosen prompt can make results worse."

Step 6: Implementing Audio Chunking

In [27]:
chunk_length_ms = 60 * 1000       # 60 seconds, in milliseconds
# For real meetings use: chunk_length_ms = 10 * 60 * 1000   (10 minutes)

os.makedirs("chunks", exist_ok=True)

In [28]:
chunks = []

for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):
    chunk = audio[start_ms:start_ms + chunk_length_ms]
    chunk_path = f"chunks/chunk_{i:03d}.mp3"
    chunk.export(chunk_path, format="mp3")

    chunks.append({
        "index": i,
        "path": chunk_path,
        "start_ms": start_ms,
        "duration_ms": len(chunk),
    })

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [29]:
for c in chunks:
    print(f"Chunk {c['index']}: starts at {c['start_ms'] / 1000:.0f}s, "
          f"length {c['duration_ms'] / 1000:.1f}s, file {c['path']}")

total = sum(c["duration_ms"] for c in chunks)
print(f"\nTotal chunk time: {total / 1000:.1f}s (original: {len(audio) / 1000:.1f}s)")

Chunk 0: starts at 0s, length 60.0s, file chunks/chunk_000.mp3
Chunk 1: starts at 60s, length 60.0s, file chunks/chunk_001.mp3
Chunk 2: starts at 120s, length 13.0s, file chunks/chunk_002.mp3

Total chunk time: 133.0s (original: 133.0s)


Step 7: Transcribing Chunks with Timestamps

In [30]:
def format_time(seconds):
    """Turn seconds into HH:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

print(format_time(125))   # should print 00:02:05

00:02:05


In [31]:
all_segments = []

for c in chunks:
    with open(c["path"], "rb") as f:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    offset = c["start_ms"] / 1000       # chunk start, in seconds

    for seg in result.segments:
        all_segments.append({
            "start": seg.start + offset,
            "end": seg.end + offset,
            "text": seg.text.strip(),
        })

    print(f"Chunk {c['index']} done: {len(result.segments)} segments")

print("Total segments:", len(all_segments))

Chunk 0 done: 13 segments
Chunk 1 done: 13 segments
Chunk 2 done: 2 segments
Total segments: 28


In [32]:
for seg in all_segments:
    print(f"[{format_time(seg['start'])} - {format_time(seg['end'])}] {seg['text']}")

[00:00:00 - 00:00:06] Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here.
[00:00:06 - 00:00:12] Well, now, I don't remember the day or the month or nothing like that, you understand?
[00:00:12 - 00:00:19] But I know they come one evening, and I just was fixing to quit working in my garden.
[00:00:19 - 00:00:27] And he come, and he says, well, says, is this Miss Davis?
[00:00:27 - 00:00:30] And I said, that's what they called me.
[00:00:30 - 00:00:34] Other words ain't new, I said, yes.
[00:00:34 - 00:00:39] He says, well, says, I'm taking pictures around.
[00:00:39 - 00:00:46] Says the government's going to take over everything, and Jamesworth's going to have to move out.
[00:00:46 - 00:00:48] I said, all this many years?
[00:00:48 - 00:00:50] I said, you talk crazy.
[00:00:50 - 00:00:52] And he lay.
[00:00:52 - 00:00:54] He says, no.
[00:00:54 - 00:00:59] Says, I'm here just to tell you, says, friendly talk.
[00:01:00 - 00:

In [33]:
with open("transcript_with_timestamps.txt", "w", encoding="utf-8") as f:
    for seg in all_segments:
        f.write(f"[{format_time(seg['start'])} - {format_time(seg['end'])}] {seg['text']}\n")

full_text = " ".join(seg["text"] for seg in all_segments)
print("Words in full transcript:", len(full_text.split()))
print("Last segment ends at:", round(all_segments[-1]["end"], 1), "seconds")

Words in full transcript: 326
Last segment ends at: 130.1 seconds


Step 8: Exporting with Timestamps

In [34]:
with open("transcript.txt", "w", encoding="utf-8") as f:
    f.write("MEETING TRANSCRIPT\n")
    f.write(f"Source: {audio_path}\n")
    f.write(f"Duration: {format_time(len(audio) / 1000)}\n")
    f.write("=" * 50 + "\n\n")
    for seg in all_segments:
        f.write(f"[{format_time(seg['start'])}] {seg['text']}\n")

print("Saved transcript.txt")

Saved transcript.txt


In [35]:
def format_srt_time(seconds):
    """Turn seconds into the SRT format HH:MM:SS,mmm."""
    total_ms = int(round(seconds * 1000))
    h = total_ms // 3_600_000
    m = (total_ms % 3_600_000) // 60_000
    s = (total_ms % 60_000) // 1000
    ms = total_ms % 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

with open("transcript.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(all_segments, start=1):
        f.write(f"{i}\n")
        f.write(f"{format_srt_time(seg['start'])} --> {format_srt_time(seg['end'])}\n")
        f.write(f"{seg['text']}\n\n")

print("Saved transcript.srt")

Saved transcript.srt


In [36]:
import json

export_data = {
    "source_file": audio_path,
    "duration_seconds": round(len(audio) / 1000, 1),
    "chunk_length_seconds": chunk_length_ms / 1000,
    "segments": [
        {
            "start": round(seg["start"], 2),
            "end": round(seg["end"], 2),
            "start_formatted": format_time(seg["start"]),
            "text": seg["text"],
        }
        for seg in all_segments
    ],
    "full_text": " ".join(seg["text"] for seg in all_segments),
}

with open("transcript.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print("Saved transcript.json")

Saved transcript.json


In [37]:
for name in ["transcript.txt", "transcript.srt", "transcript.json"]:
    print(f"{name}: {os.path.getsize(name)} bytes")

print("\n--- First lines of transcript.srt ---")
with open("transcript.srt", encoding="utf-8") as f:
    print(f.read()[:400])

transcript.txt: 2077 bytes
transcript.srt: 2674 bytes
transcript.json: 6534 bytes

--- First lines of transcript.srt ---
1
00:00:00,000 --> 00:00:06,000
Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here.

2
00:00:06,000 --> 00:00:12,000
Well, now, I don't remember the day or the month or nothing like that, you understand?

3
00:00:12,000 --> 00:00:19,000
But I know they come one evening, and I just was fixing to quit working in my garden.

4
00:00:19,000


In [38]:
keyword = "garden"
for seg in export_data["segments"]:
    if keyword.lower() in seg["text"].lower():
        print(f"[{seg['start_formatted']}] {seg['text']}")

[00:00:12] But I know they come one evening, and I just was fixing to quit working in my garden.
